# Challenge Dataset Generation
Generates CheckList-style SRL test instances for 6 capabilities using GPT-4o.
Each capability has 3 cells: **Generate → Edit → Save**.

In [ ]:
import os, json
import pandas as pd
from openai import OpenAI

'''
**Note:** Set your OpenAI API key as an environment variable before running:s
export OPENAI_API_KEY="your-key-here"
'''

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

os.makedirs("challenge_dataset", exist_ok=True)

def ask_gpt(prompt):
    r = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    raw = r.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    return json.loads(raw.strip())

def show(instances):
    rows = []
    for x in instances:
        sent = x["sentence"]
        rows.append({
            "id": x["id"],
            "sentence": " ".join(sent),
            "predicate_tok": sent[x["predicate"].index(1)],
            "target_tok": sent[x["target_index"]],
            "target_label": x["target_label"],
            "type": x["type"],
            "pair_id": x.get("pair_id", ""),
            "role": x.get("role", ""),
        })
    return pd.DataFrame(rows)

def save(instances, filename):
    errors = []
    for x in instances:
        if len(x["sentence"]) != len(x["predicate"]):
            errors.append(f"{x['id']}: sentence/predicate length mismatch")
        if sum(x["predicate"]) != 1:
            errors.append(f"{x['id']}: predicate must have exactly one 1")
        if not (0 <= x["target_index"] < len(x["sentence"])):
            errors.append(f"{x['id']}: target_index out of range")
    if errors:
        print("Fix these before saving:")
        for e in errors: print(" ", e)
    else:
        path = f"challenge_dataset/{filename}"
        with open(path, "w") as f: json.dump(instances, f, indent=2)
        print(f"Saved {len(instances)} instances -> {path}")

print("Ready.")

Ready.


---
## Capability 1 — Core Argument Identification (MFT)
Tests that both models correctly label ARG0 (agent/subject) and ARG1 (patient/object) in simple active sentences. This is the sanity check, both models should pass.

In [2]:
# GENERATE
prompt_cap1 = """Generate exactly 10 unique sentences for SRL testing.
All 10 sentences must be distinct — no repeated subject-verb-object combinations.
For each sentence produce TWO instances (one ARG0, one ARG1). Output 20 objects total.

Sentence requirements across the 10 sentences:
- Active declarative SVO structure only. No passives, no modals, no negation.
- At least 3 subjects that are NOT simple person names (organizations, role nouns, animals, collectives)
- At least 3 objects that are modified NPs (e.g. 'the damaged report', 'three prototype engines')
- No repeated verbs across the 10 sentences
- Mix of sentence lengths: 6, 7, 8, and 9 tokens (punctuation is its own token)
- At least 2 sentences with an adjunct (manner/temporal/locative) that does not change ARG0/ARG1

Required JSON fields per instance:
- id: "cap1a_001" for ARG0, "cap1b_001" for ARG1 (pair shares the same number)
- capability: 1
- test: "core_arg0" or "core_arg1"
- type: "MFT"
- sentence: list of string tokens
- predicate: list of ints, same length as sentence, exactly one 1 at the verb
- target_index: 0-based int pointing to ARG0 (subject head) or ARG1 (object head)
- target_label: "ARG0" or "ARG1"

Example pair:
[{"id": "cap1a_001", "capability": 1, "test": "core_arg0", "type": "MFT",
  "sentence": ["The", "surgeon", "carefully", "removed", "the", "tumor", "."],
  "predicate": [0, 0, 0, 1, 0, 0, 0], "target_index": 1, "target_label": "ARG0"},
 {"id": "cap1b_001", "capability": 1, "test": "core_arg1", "type": "MFT",
  "sentence": ["The", "surgeon", "carefully", "removed", "the", "tumor", "."],
  "predicate": [0, 0, 0, 1, 0, 0, 0], "target_index": 5, "target_label": "ARG1"}]

Output ONLY the JSON array."""

cap1 = ask_gpt(prompt_cap1)
show(cap1)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap1a_001,The committee approved the new policy .,approved,committee,ARG0,MFT,,
1,cap1b_001,The committee approved the new policy .,approved,policy,ARG1,MFT,,
2,cap1a_002,A swarm of bees attacked the intruder .,attacked,swarm,ARG0,MFT,,
3,cap1b_002,A swarm of bees attacked the intruder .,attacked,intruder,ARG1,MFT,,
4,cap1a_003,The engineer designed three prototype engines .,designed,engineer,ARG0,MFT,,
5,cap1b_003,The engineer designed three prototype engines .,designed,prototype,ARG1,MFT,,
6,cap1a_004,The teacher explained the complex concept clea...,explained,teacher,ARG0,MFT,,
7,cap1b_004,The teacher explained the complex concept clea...,explained,concept,ARG1,MFT,,
8,cap1a_005,The dog chased the red ball quickly .,chased,dog,ARG0,MFT,,
9,cap1b_005,The dog chased the red ball quickly .,chased,ball,ARG1,MFT,,


### Note on generated results
Cap 1 — Clean. All indices correct, no duplicates, good variety. One minor note: cap1_002 target for ARG0 is index 1 ("swarm") not index 3 ("bees"), that's correct according to PropBank-style (head of the NP) annotation. 

In [ ]:
# EDIT — fix anything wrong here before saving
# cap1[i]["target_index"] = correct_value
# cap1[i]["sentence"] = ["New", "sentence", "."]
# cap1.pop(i)  # delete an instance

# --- edits ---


# --- end edits ---
show(cap1)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap1a_001,The committee approved the new policy .,approved,committee,ARG0,MFT,,
1,cap1b_001,The committee approved the new policy .,approved,policy,ARG1,MFT,,
2,cap1a_002,A swarm of bees attacked the intruder .,attacked,swarm,ARG0,MFT,,
3,cap1b_002,A swarm of bees attacked the intruder .,attacked,intruder,ARG1,MFT,,
4,cap1a_003,The engineer designed three prototype engines .,designed,engineer,ARG0,MFT,,
5,cap1b_003,The engineer designed three prototype engines .,designed,prototype,ARG1,MFT,,
6,cap1a_004,The teacher explained the complex concept clea...,explained,teacher,ARG0,MFT,,
7,cap1b_004,The teacher explained the complex concept clea...,explained,concept,ARG1,MFT,,
8,cap1a_005,The dog chased the red ball quickly .,chased,dog,ARG0,MFT,,
9,cap1b_005,The dog chased the red ball quickly .,chased,ball,ARG1,MFT,,


In [4]:
# SAVE
save(cap1, "cap1_core_arguments.json")

Saved 20 instances -> challenge_dataset/cap1_core_arguments.json


---
## Capability 2 — Voice Alternation (INV)
Tests that the agent (ARG0) is correctly identified regardless of voice. In active sentences ARG0 is the subject. In passive sentences ARG0 is the by-phrase agent. LR (Logistic Regression) is expected to fail this, its dependency path features look completely different for active subjects vs passive by-phrases. (destil)BERT should handle it better via contextual understanding.

In [5]:
# GENERATE
prompt_cap2 = """Generate exactly 10 active/passive sentence PAIRS for SRL voice alternation testing.
All 10 pairs must be distinct — no repeated verbs or agent-patient combinations.
Output 20 objects total (10 active + 10 passive).

For each pair:
- Active: "The chef cooked the fish." -> agent is the subject -> target_label ARG0
- Passive: "The fish was cooked by the chef." -> agent is the by-phrase noun -> target_label ARG0
- Both instances test the SAME entity (the agent) getting ARG0
- Always include an explicit "by [agent]" — no agentless passives

Variety requirements:
- Use clear transitive verbs with explicit agents
- Mix agent types: person names, role nouns, organizations, collectives
- No repeated verbs across the 10 pairs
- Sentence length 6-10 tokens (punctuation is its own token)
- predicate in passive points to the PAST PARTICIPLE (not the auxiliary 'was')
- target_index in passive points to the noun AFTER 'by' (not to 'by' itself)

Required JSON fields per instance:
- id: "cap2_001a" (active) / "cap2_001b" (passive)
- pair_id: "cap2_001" (same for both)
- role: "active" or "passive"
- capability: 2
- test: "voice_alternation"
- type: "INV"
- sentence: list of string tokens
- predicate: list of ints, exactly one 1
- target_index: 0-based int pointing to the agent noun
- target_label: "ARG0"

Example pair:
[{"id": "cap2_001a", "pair_id": "cap2_001", "role": "active",
  "capability": 2, "test": "voice_alternation", "type": "INV",
  "sentence": ["The", "chef", "cooked", "the", "fish", "."],
  "predicate": [0, 0, 1, 0, 0, 0], "target_index": 1, "target_label": "ARG0"},
 {"id": "cap2_001b", "pair_id": "cap2_001", "role": "passive",
  "capability": 2, "test": "voice_alternation", "type": "INV",
  "sentence": ["The", "fish", "was", "cooked", "by", "the", "chef", "."],
  "predicate": [0, 0, 0, 1, 0, 0, 0, 0], "target_index": 6, "target_label": "ARG0"}]

Output ONLY the JSON array."""

cap2 = ask_gpt(prompt_cap2)
show(cap2)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap2_001a,The teacher praised the student .,praised,teacher,ARG0,INV,cap2_001,active
1,cap2_001b,The student was praised by the teacher .,praised,teacher,ARG0,INV,cap2_001,passive
2,cap2_002a,The manager approved the budget .,approved,manager,ARG0,INV,cap2_002,active
3,cap2_002b,The budget was approved by the manager .,approved,manager,ARG0,INV,cap2_002,passive
4,cap2_003a,The artist painted a mural .,painted,artist,ARG0,INV,cap2_003,active
5,cap2_003b,A mural was painted by the artist .,painted,artist,ARG0,INV,cap2_003,passive
6,cap2_004a,The scientist discovered a cure .,discovered,scientist,ARG0,INV,cap2_004,active
7,cap2_004b,A cure was discovered by the scientist .,discovered,scientist,ARG0,INV,cap2_004,passive
8,cap2_005a,The author wrote a novel .,wrote,author,ARG0,INV,cap2_005,active
9,cap2_005b,A novel was written by the author .,written,author,ARG0,INV,cap2_005,passive


### Note on generated results

Cap 2 — Clean. All passive target_index correctly point to the agent noun after "by" (index 6 in all cases), predicate correctly points to the past participle not "was". Pairs consistent.

In [ ]:
# EDIT
# Main things to check:
# - target_index in passive points to the agent noun, NOT to "by"
# - predicate in passive points to the past participle, NOT to "was"
# - pair_id matches within each pair

# --- edits ---


# --- end edits ---
show(cap2)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap2_001a,The teacher praised the student .,praised,teacher,ARG0,INV,cap2_001,active
1,cap2_001b,The student was praised by the teacher .,praised,teacher,ARG0,INV,cap2_001,passive
2,cap2_002a,The manager approved the budget .,approved,manager,ARG0,INV,cap2_002,active
3,cap2_002b,The budget was approved by the manager .,approved,manager,ARG0,INV,cap2_002,passive
4,cap2_003a,The artist painted a mural .,painted,artist,ARG0,INV,cap2_003,active
5,cap2_003b,A mural was painted by the artist .,painted,artist,ARG0,INV,cap2_003,passive
6,cap2_004a,The scientist discovered a cure .,discovered,scientist,ARG0,INV,cap2_004,active
7,cap2_004b,A cure was discovered by the scientist .,discovered,scientist,ARG0,INV,cap2_004,passive
8,cap2_005a,The author wrote a novel .,wrote,author,ARG0,INV,cap2_005,active
9,cap2_005b,A novel was written by the author .,written,author,ARG0,INV,cap2_005,passive


In [7]:
# SAVE
save(cap2, "cap2_voice_alternation.json")

Saved 20 instances -> challenge_dataset/cap2_voice_alternation.json


---
## Capability 3 — PP Attachment Ambiguity / "with X" (MFT)
Tests whether models can disambiguate the same surface form "with X" into different semantic roles based on the semantics of X:
- Instrument -> ARGM-MNR ("with a spoon")
- Comitative -> ARGM-COM ("with her friend"), expected to be hard for both models due to label rarity
- Incorporated patient -> ARG1 ("with paint", for spray/fill-type verbs)

LR relies on syntax and cannot distinguish these. BERT should do better via lexical semantics.

In [6]:
# GENERATE
prompt_cap3 = """Generate exactly 15 SRL test instances for PP attachment ambiguity testing.
All 15 sentences must be distinct — no repeated verb-noun combinations.

The test uses sentences with a prepositional phrase starting with "with".
The semantic role of the with-phrase head noun differs by context:

Type 1 — ARGM-MNR (instrument): the with-phrase describes HOW the action is done using a tool
  Example: "She ate the soup with a spoon." -> target = "spoon", label = ARGM-MNR
  Use inanimate tool/instrument nouns. Verbs: eat, stir, cut, write, open, fix, dig

Type 2 — ARGM-COM (comitative): the with-phrase describes WHO accompanied the agent
  Example: "She ate the soup with her friend." -> target = "friend", label = ARGM-COM
  Use animate companion nouns (person names, role nouns). Verbs: eat, walk, travel, work, celebrate

Type 3 — ARG2 (substance/theme): spray/load/fill verbs where the with-phrase is the SECOND
  core argument (the substance being moved into the container/location).
  In PropBank, ARG1 = the container (direct object), ARG2 = the substance (with-phrase).
  Example: "She filled the glass with water." -> target = "water", label = ARG2
  Only use these verbs: fill, load, spray, cover, stuff, pack, flood

Generate 5 instances of each type (15 total). Vary verbs and nouns across all instances.
target_index must point to the HEAD NOUN of the with-phrase (NOT to the word 'with').
Sentence length 7-12 tokens (punctuation is its own token).

Required JSON fields:
- id: "cap3_mnr_001", "cap3_com_001", "cap3_arg2_001"
- capability: 3
- test: "pp_attachment_mnr", "pp_attachment_com", or "pp_attachment_arg2"
- type: "MFT"
- sentence: list of string tokens
- predicate: list of ints, exactly one 1 at the main verb
- target_index: 0-based int pointing to the HEAD NOUN of the with-phrase
- target_label: "ARGM-MNR", "ARGM-COM", or "ARG2"

Example:
{"id": "cap3_mnr_001", "capability": 3, "test": "pp_attachment_mnr", "type": "MFT",
 "sentence": ["She", "stirred", "the", "sauce", "with", "a", "wooden", "spoon", "."],
 "predicate": [0, 1, 0, 0, 0, 0, 0, 0, 0], "target_index": 7, "target_label": "ARGM-MNR"}

Output ONLY the JSON array."""

cap3 = ask_gpt(prompt_cap3)
show(cap3)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap3_mnr_001,He cut the paper with scissors .,cut,scissors,ARGM-MNR,MFT,,
1,cap3_mnr_002,They opened the box with a crowbar .,opened,crowbar,ARGM-MNR,MFT,,
2,cap3_mnr_003,She wrote the letter with a pen .,wrote,pen,ARGM-MNR,MFT,,
3,cap3_mnr_004,We fixed the leak with duct tape .,fixed,tape,ARGM-MNR,MFT,,
4,cap3_mnr_005,He dug the hole with a shovel .,dug,shovel,ARGM-MNR,MFT,,
5,cap3_com_001,She walked to the park with her brother .,walked,brother,ARGM-COM,MFT,,
6,cap3_com_002,They celebrated the holiday with friends .,celebrated,friends,ARGM-COM,MFT,,
7,cap3_com_003,He ate dinner with his colleague .,ate,colleague,ARGM-COM,MFT,,
8,cap3_com_004,We traveled to Italy with our family .,traveled,family,ARGM-COM,MFT,,
9,cap3_com_005,She worked on the project with a team .,worked,team,ARGM-COM,MFT,,


### Note on generated results

Cap 3 — Clean. All target_index correctly on the head noun, never on "with". Good verb choices for each type. ARGM-COM instances look linguistically valid.

In [17]:
# EDIT
# Main things to check:
# - target_index points to the HEAD NOUN, not to the word "with"
# - ARG1 instances use spray/fill/load-type verbs only
# - ARGM-COM companions are animate (people/animals), not instruments
# - ARGM-MNR instruments are inanimate tools

# --- edits ---
for inst in cap3:
    if inst["id"] == "cap3_com_004":
        inst["predicate"] = [0, 1, 0, 0, 0, 0, 0, 0]  # 8 ints to match 8 tokens

# --- end edits ---
show(cap3)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap3_mnr_001,He cut the paper with scissors .,cut,scissors,ARGM-MNR,MFT,,
1,cap3_mnr_002,They opened the box with a crowbar .,opened,crowbar,ARGM-MNR,MFT,,
2,cap3_mnr_003,She wrote the letter with a pen .,wrote,pen,ARGM-MNR,MFT,,
3,cap3_mnr_004,We fixed the leak with duct tape .,fixed,tape,ARGM-MNR,MFT,,
4,cap3_mnr_005,He dug the hole with a shovel .,dug,shovel,ARGM-MNR,MFT,,
5,cap3_com_001,She walked to the park with her brother .,walked,brother,ARGM-COM,MFT,,
6,cap3_com_002,They celebrated the holiday with friends .,celebrated,friends,ARGM-COM,MFT,,
7,cap3_com_003,He ate dinner with his colleague .,ate,colleague,ARGM-COM,MFT,,
8,cap3_com_004,We traveled to Italy with our family .,traveled,family,ARGM-COM,MFT,,
9,cap3_com_005,She worked on the project with a team .,worked,team,ARGM-COM,MFT,,


In [19]:
# SAVE
save(cap3, "cap3_pp_attachment.json")

Saved 15 instances -> challenge_dataset/cap3_pp_attachment.json


---
## Capability 4 — ARGM-GOL Detection (MFT)
Tests whether models can identify goal arguments (destinations/endpoints of motion or transfer events). A1 scored F1=0.00 on ARGM-GOL — it should fail almost entirely. A2 performance unknown but expected to be better.

Key distinction: ARGM-GOL is the endpoint of movement (where something *goes to*), not ARGM-LOC (where something *happens at*).

In [11]:
# GENERATE
prompt_cap4 = """Generate exactly 10 SRL test instances for ARGM-GOL (goal argument) detection.
All 10 sentences must be distinct — no repeated verb-destination combinations.

ARGM-GOL marks the endpoint/destination of a motion or transfer event.
It is introduced by prepositions: "to", "into", "onto", "toward", "towards".

Use these verb types:
- Motion verbs: walk, run, drive, fly, swim, travel, move, march, climb, rush
- Transfer verbs: send, deliver, bring, carry, ship, give, pass, hand, transport

Critical distinction — do NOT confuse with ARGM-LOC:
- ARGM-GOL = endpoint of movement: "She walked TO the office" (she ends up there)
- ARGM-LOC = static location: "She worked AT the office" (she is already there)
- Only use verbs that describe movement or transfer, not states

target_index must point to the HEAD NOUN of the goal phrase (NOT to the preposition).
Vary destinations: cities, buildings, rooms, people, organizations.
Sentence length 7-11 tokens (punctuation is its own token).

Required JSON fields:
- id: "cap4_001"
- capability: 4
- test: "argm_gol_detection"
- type: "MFT"
- sentence: list of string tokens
- predicate: list of ints, exactly one 1 at the main verb
- target_index: 0-based int pointing to HEAD NOUN of the goal phrase
- target_label: "ARGM-GOL"

Example:
{"id": "cap4_001", "capability": 4, "test": "argm_gol_detection", "type": "MFT",
 "sentence": ["She", "sent", "the", "package", "to", "her", "sister", "."],
 "predicate": [0, 1, 0, 0, 0, 0, 0, 0], "target_index": 6, "target_label": "ARGM-GOL"}

Output ONLY the JSON array."""

cap4 = ask_gpt(prompt_cap4)
show(cap4)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap4_002,They drove to the airport for their flight .,drove,airport,ARGM-GOL,MFT,,
1,cap4_003,She ran toward the park to meet friends .,ran,park,ARGM-GOL,MFT,,
2,cap4_004,The letter was delivered to the company office .,delivered,office,ARGM-GOL,MFT,,
3,cap4_005,We flew into New York City last night .,flew,York,ARGM-GOL,MFT,,
4,cap4_006,He swam towards the shore quickly to avoid the...,swam,shore,ARGM-GOL,MFT,,
5,cap4_007,The shipment was sent to the warehouse yesterd...,sent,warehouse,ARGM-GOL,MFT,,
6,cap4_008,I moved the furniture into the living room tod...,moved,room,ARGM-GOL,MFT,,
7,cap4_009,The troops marched toward the fortress at dawn .,marched,fortress,ARGM-GOL,MFT,,
8,cap4_010,She climbed onto the roof to fix the antenna .,climbed,roof,ARGM-GOL,MFT,,
9,cap4_011,They rushed to the hospital after the accident .,rushed,hospital,ARGM-GOL,MFT,,


### Note on generated results

Cap 4 — One issue, cap4_004 is a passive sentence:

"The letter was delivered to the company office."
The prompt specified active sentences only. More importantly, the predicate points to "delivered" (index 3) which is correct, but this is the only passive in the set which is inconsistent with the rest. I replaced it with an active equivalent in Cell B:

Also cap4_005 — "New York City" is three tokens but target_index is 4 which points to "York" not "City". Made "City" is the head noun.

In [22]:
# EDIT
# Main things to check:
# - target_index points to the HEAD NOUN, not the preposition
# - Verb is genuinely a motion or transfer verb (not a static verb)
# - The phrase is a GOAL (endpoint) not a LOCATION (static)
#   Reject: "She worked in the office" (static location)
#   Keep:   "She walked into the office" (endpoint of motion)

# --- edits ---
cap4[2]["sentence"] = ["The", "courier", "delivered", "the", "letter", "to", "the", "office", "."]
cap4[2]["predicate"] = [0, 0, 1, 0, 0, 0, 0, 0, 0]
cap4[2]["target_index"] = 7  # "office" = ARGM-GOL
cap4[2]["id"] = "cap4_004"

cap4[3]["target_index"] = 5  # "City" is the head of "New York City"


# --- end edits ---
show(cap4)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap4_002,They drove to the airport for their flight .,drove,airport,ARGM-GOL,MFT,,
1,cap4_003,She ran toward the park to meet friends .,ran,park,ARGM-GOL,MFT,,
2,cap4_004,The courier delivered the letter to the office .,delivered,office,ARGM-GOL,MFT,,
3,cap4_005,We flew into New York City last night .,flew,City,ARGM-GOL,MFT,,
4,cap4_006,He swam towards the shore quickly to avoid the...,swam,shore,ARGM-GOL,MFT,,
5,cap4_007,The shipment was sent to the warehouse yesterd...,sent,warehouse,ARGM-GOL,MFT,,
6,cap4_008,I moved the furniture into the living room tod...,moved,room,ARGM-GOL,MFT,,
7,cap4_009,The troops marched toward the fortress at dawn .,marched,fortress,ARGM-GOL,MFT,,
8,cap4_010,She climbed onto the roof to fix the antenna .,climbed,roof,ARGM-GOL,MFT,,
9,cap4_011,They rushed to the hospital after the accident .,rushed,hospital,ARGM-GOL,MFT,,


In [23]:
# SAVE
save(cap4, "cap4_argm_gol.json")

Saved 10 instances -> challenge_dataset/cap4_argm_gol.json


---
## Capability 5 — Spray/Load Alternation (INV)
Tests argument alternation with spray/load verbs (spray, load, fill, cover, pack, stuff).
The same two participants swap grammatical positions across the two variants:
- Variant A (location as object): "She sprayed the wall with paint." -> paint=ARG2, wall=ARG1
- Variant B (theme as object): "She sprayed paint onto the wall." -> paint=ARG1, wall=ARGM-GOL

We test the THEME (the substance) in both variants. Its label changes from ARG2 to ARG1 across the alternation. LR will likely fail this because the surface position and dependency path change completely.

In [14]:
# GENERATE
prompt_cap5 = """Generate exactly 10 spray/load alternation PAIRS for SRL testing.
All 10 pairs must be distinct — use at least 4 different spray/load verbs.
Output 20 objects total (10 pairs x 2 variants).

Spray/load verbs participate in an alternation where two arguments swap positions:

Variant A — location as direct object:
  "She sprayed the wall with paint."
  -> wall=ARG1 (location/container), paint=ARG2 (theme/substance in with-phrase)
  -> test the THEME: target_index points to "paint", target_label = "ARG2"

Variant B — theme as direct object:
  "She sprayed paint onto the wall."
  -> paint=ARG1 (theme is now the direct object), wall=ARGM-GOL
  -> test the THEME: target_index points to "paint", target_label = "ARG1"

Use only these verbs: spray, load, fill, cover, pack, stuff, pile, flood, coat, smear
Vary agents, locations, and substances. Sentence length 7-10 tokens.
The SAME theme entity must appear in both variants of each pair.

Required JSON fields:
- id: "cap5_001a" (variant A) / "cap5_001b" (variant B)
- pair_id: "cap5_001"
- role: "location_as_object" or "theme_as_object"
- capability: 5
- test: "spray_load_alternation"
- type: "INV"
- sentence: list of string tokens
- predicate: list of ints, exactly one 1
- target_index: 0-based int pointing to the THEME noun
- target_label: "ARG2" for variant A, "ARG1" for variant B

Example pair:
[{"id": "cap5_001a", "pair_id": "cap5_001", "role": "location_as_object",
  "capability": 5, "test": "spray_load_alternation", "type": "INV",
  "sentence": ["She", "sprayed", "the", "wall", "with", "paint", "."],
  "predicate": [0, 1, 0, 0, 0, 0, 0], "target_index": 5, "target_label": "ARG2"},
 {"id": "cap5_001b", "pair_id": "cap5_001", "role": "theme_as_object",
  "capability": 5, "test": "spray_load_alternation", "type": "INV",
  "sentence": ["She", "sprayed", "paint", "onto", "the", "wall", "."],
  "predicate": [0, 1, 0, 0, 0, 0, 0], "target_index": 2, "target_label": "ARG1"}]

Output ONLY the JSON array."""

cap5 = ask_gpt(prompt_cap5)
show(cap5)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap5_001a,He loaded the truck with boxes .,loaded,boxes,ARG2,INV,cap5_001,location_as_object
1,cap5_001b,He loaded boxes into the truck .,loaded,boxes,ARG1,INV,cap5_001,theme_as_object
2,cap5_002a,They filled the bottle with water .,filled,water,ARG2,INV,cap5_002,location_as_object
3,cap5_002b,They filled water into the bottle .,filled,water,ARG1,INV,cap5_002,theme_as_object
4,cap5_003a,She covered the cake with frosting .,covered,frosting,ARG2,INV,cap5_003,location_as_object
5,cap5_003b,She covered frosting onto the cake .,covered,frosting,ARG1,INV,cap5_003,theme_as_object
6,cap5_004a,He packed the suitcase with clothes .,packed,clothes,ARG2,INV,cap5_004,location_as_object
7,cap5_004b,He packed clothes into the suitcase .,packed,clothes,ARG1,INV,cap5_004,theme_as_object
8,cap5_005a,They stuffed the pillow with feathers .,stuffed,feathers,ARG2,INV,cap5_005,location_as_object
9,cap5_005b,They stuffed feathers into the pillow .,stuffed,feathers,ARG1,INV,cap5_005,theme_as_object


### Note on generated results

Cap 5 — One linguistic note. 

cap5_002b "They filled water into the bottle" is slightly unnatural English, native speakers would say "They poured water into the bottle". The alternation holds semantically but I swapped the verb.

Also cap5_007b "He flooded water into the garden", same thing. "Flooded" doesn't feel natural here. Changed the B variant.

In [24]:
# EDIT
# Main things to check:
# - target_index points to the THEME noun (the substance), not the location
# - Variant A: theme is in the with-phrase -> target_label = ARG2
# - Variant B: theme is the direct object -> target_label = ARG1
# - The same theme word appears in both members of each pair
# - pair_id matches within each pair

# --- edits ---
cap5[3]["sentence"] = ["They", "poured", "water", "into", "the", "bottle", "."]
cap5[3]["predicate"] = [0, 1, 0, 0, 0, 0, 0]

cap5[13]["sentence"] = ["He", "poured", "water", "into", "the", "garden", "."]
cap5[13]["predicate"] = [0, 1, 0, 0, 0, 0, 0]

# --- end edits ---
show(cap5)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap5_001a,He loaded the truck with boxes .,loaded,boxes,ARG2,INV,cap5_001,location_as_object
1,cap5_001b,He loaded boxes into the truck .,loaded,boxes,ARG1,INV,cap5_001,theme_as_object
2,cap5_002a,They filled the bottle with water .,filled,water,ARG2,INV,cap5_002,location_as_object
3,cap5_002b,They poured water into the bottle .,poured,water,ARG1,INV,cap5_002,theme_as_object
4,cap5_003a,She covered the cake with frosting .,covered,frosting,ARG2,INV,cap5_003,location_as_object
5,cap5_003b,She covered frosting onto the cake .,covered,frosting,ARG1,INV,cap5_003,theme_as_object
6,cap5_004a,He packed the suitcase with clothes .,packed,clothes,ARG2,INV,cap5_004,location_as_object
7,cap5_004b,He packed clothes into the suitcase .,packed,clothes,ARG1,INV,cap5_004,theme_as_object
8,cap5_005a,They stuffed the pillow with feathers .,stuffed,feathers,ARG2,INV,cap5_005,location_as_object
9,cap5_005b,They stuffed feathers into the pillow .,stuffed,feathers,ARG1,INV,cap5_005,theme_as_object


In [25]:
# SAVE
save(cap5, "cap5_spray_load.json")

Saved 20 instances -> challenge_dataset/cap5_spray_load.json


---
## Capability 6 — Named Entity Substitution (INV)
Tests robustness to unseen proper nouns. Both variants are identical except the ARG0 agent name is swapped: a common English name vs a rare/foreign name. Both should still be labeled ARG0. LR may fail on unseen entity names (token lemma feature). BERT should be robust via context.

In [17]:
# GENERATE
prompt_cap6 = """Generate exactly 10 named entity substitution PAIRS for SRL testing.
All 10 pairs must be distinct — no repeated sentence structures or verb-object combinations.
Output 20 objects total (10 pairs x 2 variants).

Each pair = two sentences IDENTICAL in structure, differing ONLY in the agent name:
- Variant A: common English name as agent (John, Mary, Smith Corp, Apple Inc)
- Variant B: rare/foreign/unusual name as agent that would be uncommon in English NLP training data
  Use names from: West African (Babatunde, Amara, Kofi), Eastern European (Przemek, Zbigniew, Ewa),
  Chinese (Xiulan, Weiming, Fenghua), Arabic (Tariq, Nourhan, Suhaib),
  or clearly fictional-sounding names

Rules:
- Active declarative SVO sentences only
- Agent is always the sentence subject (position 0 or after 'The')
- Vary verbs and objects — no repeated verbs across the 10 pairs
- Include a mix of individual person names and organization names
- Sentence length 5-9 tokens (punctuation is its own token)
- target_index must point to the NAME TOKEN (the proper noun itself)
- Both members of each pair have identical predicate and target_index

Required JSON fields:
- id: "cap6_001a" (common) / "cap6_001b" (rare)
- pair_id: "cap6_001"
- role: "common_name" or "rare_name"
- capability: 6
- test: "entity_substitution"
- type: "INV"
- sentence: list of string tokens
- predicate: list of ints, exactly one 1
- target_index: 0-based int pointing to the agent name token
- target_label: "ARG0"

Example pair:
[{"id": "cap6_001a", "pair_id": "cap6_001", "role": "common_name",
  "capability": 6, "test": "entity_substitution", "type": "INV",
  "sentence": ["John", "signed", "the", "contract", "."],
  "predicate": [0, 1, 0, 0, 0], "target_index": 0, "target_label": "ARG0"},
 {"id": "cap6_001b", "pair_id": "cap6_001", "role": "rare_name",
  "capability": 6, "test": "entity_substitution", "type": "INV",
  "sentence": ["Babatunde", "signed", "the", "contract", "."],
  "predicate": [0, 1, 0, 0, 0], "target_index": 0, "target_label": "ARG0"}]

Output ONLY the JSON array."""

cap6 = ask_gpt(prompt_cap6)
show(cap6)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap6_001a,Alice bought a new car .,bought,Alice,ARG0,INV,cap6_001,common_name
1,cap6_001b,Amara bought a new car .,bought,Amara,ARG0,INV,cap6_001,rare_name
2,cap6_002a,Microsoft released the new update .,released,Microsoft,ARG0,INV,cap6_002,common_name
3,cap6_002b,Fenghua released the new update .,released,Fenghua,ARG0,INV,cap6_002,rare_name
4,cap6_003a,Sarah cooked a delicious meal .,cooked,Sarah,ARG0,INV,cap6_003,common_name
5,cap6_003b,Tariq cooked a delicious meal .,cooked,Tariq,ARG0,INV,cap6_003,rare_name
6,cap6_004a,The team celebrated their victory .,celebrated,team,ARG0,INV,cap6_004,common_name
7,cap6_004b,Zbigniew celebrated their victory .,celebrated,Zbigniew,ARG0,INV,cap6_004,rare_name
8,cap6_005a,Google acquired a small startup .,acquired,Google,ARG0,INV,cap6_005,common_name
9,cap6_005b,Weiming acquired a small startup .,acquired,Weiming,ARG0,INV,cap6_005,rare_name


### Note on generated results

Cap 6 — One real problem. cap6_004 pair is broken, the two members are NOT the same sentence with only the name swapped:

004a: "The team celebrated their victory ." (6 tokens, target_index 1 = "team")
004b: "Zbigniew celebrated their victory ." (5 tokens, target_index 0 = "Zbigniew")

Different sentence structure, different length. Fixed by making them match.

In [26]:
# EDIT
# Main things to check:
# - Both members of each pair are identical except for the agent name
# - target_index points to the name token itself
# - Rare names are genuinely unusual in English NLP corpora
# - predicate and target_index are the same in both members of each pair

# --- edits ---
cap6[6]["sentence"] = ["Tom", "celebrated", "their", "victory", "."]
cap6[6]["predicate"] = [0, 1, 0, 0, 0]
cap6[6]["target_index"] = 0
cap6[6]["role"] = "common_name"
cap6[6]["id"] = "cap6_004a"

cap6[7]["sentence"] = ["Zbigniew", "celebrated", "their", "victory", "."]
cap6[7]["predicate"] = [0, 1, 0, 0, 0]
cap6[7]["target_index"] = 0

# --- end edits ---
show(cap6)

,id,sentence,predicate_tok,target_tok,target_label,type,pair_id,role
0,cap6_001a,Alice bought a new car .,bought,Alice,ARG0,INV,cap6_001,common_name
1,cap6_001b,Amara bought a new car .,bought,Amara,ARG0,INV,cap6_001,rare_name
2,cap6_002a,Microsoft released the new update .,released,Microsoft,ARG0,INV,cap6_002,common_name
3,cap6_002b,Fenghua released the new update .,released,Fenghua,ARG0,INV,cap6_002,rare_name
4,cap6_003a,Sarah cooked a delicious meal .,cooked,Sarah,ARG0,INV,cap6_003,common_name
5,cap6_003b,Tariq cooked a delicious meal .,cooked,Tariq,ARG0,INV,cap6_003,rare_name
6,cap6_004a,Tom celebrated their victory .,celebrated,Tom,ARG0,INV,cap6_004,common_name
7,cap6_004b,Zbigniew celebrated their victory .,celebrated,Zbigniew,ARG0,INV,cap6_004,rare_name
8,cap6_005a,Google acquired a small startup .,acquired,Google,ARG0,INV,cap6_005,common_name
9,cap6_005b,Weiming acquired a small startup .,acquired,Weiming,ARG0,INV,cap6_005,rare_name


In [27]:
# SAVE
save(cap6, "cap6_entity_substitution.json")

Saved 20 instances -> challenge_dataset/cap6_entity_substitution.json


---
## Summary

In [28]:
print("Challenge dataset summary")
print("=" * 50)
total = 0
for fname in sorted(os.listdir("challenge_dataset")):
    if fname.endswith(".json"):
        with open(f"challenge_dataset/{fname}") as f:
            data = json.load(f)
        n = len(data)
        total += n
        mft = sum(1 for x in data if x["type"] == "MFT")
        inv = sum(1 for x in data if x["type"] == "INV")
        print(f"{fname:<45} {n:>3} instances  (MFT: {mft}, INV: {inv})")
print("-" * 50)
print(f"Total: {total} instances")

Challenge dataset summary
cap1_core_arguments.json                       20 instances  (MFT: 20, INV: 0)
cap2_voice_alternation.json                    20 instances  (MFT: 0, INV: 20)
cap3_pp_attachment.json                        15 instances  (MFT: 15, INV: 0)
cap4_argm_gol.json                             10 instances  (MFT: 10, INV: 0)
cap5_spray_load.json                           20 instances  (MFT: 0, INV: 20)
cap6_entity_substitution.json                  20 instances  (MFT: 0, INV: 20)
--------------------------------------------------
Total: 105 instances
